# 02 — Preparação da base de conhecimento

## Objetivo
Carregar as tabelas relacionais necessárias, limpar as avaliações, enriquecer seus metadados e transformá-las em documentos para o RAG.

A lógica reutilizada está em `src/data/loader.py` e `src/data/preprocessing.py`.

**Princípio de modelagem:** uma avaliação com comentário representa um documento semântico. As tabelas de itens, produtos, vendedores e pagamentos são agregadas por pedido para não multiplicar artificialmente as avaliações.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
from src.data.loader import load_olist_raw
from src.data.preprocessing import prepare_reviews, enrich_reviews, build_documents

raw = load_olist_raw(RAW_DIR)
print("Tabelas carregadas:")
for name, frame in raw.items():
    print(f"- {name}: {frame.shape[0]:,} linhas x {frame.shape[1]} colunas")


Tabelas carregadas:
- reviews: 99,224 linhas x 7 colunas
- orders: 99,441 linhas x 8 colunas
- items: 112,650 linhas x 7 colunas
- customers: 99,441 linhas x 5 colunas
- products: 32,951 linhas x 9 colunas
- sellers: 3,095 linhas x 4 colunas
- payments: 103,886 linhas x 5 colunas
- categories: 71 linhas x 2 colunas


## 1. Qualidade da tabela de avaliações

Compare a tabela original com o conjunto após limpeza textual.

In [2]:
reviews_raw = raw["reviews"]
reviews_clean = prepare_reviews(reviews_raw)

print(f"Avaliações originais: {len(reviews_raw):,}")
print(f"Avaliações com comentário válido: {len(reviews_clean):,}")
print(f"Registros removidos por comentário vazio: {len(reviews_raw) - len(reviews_clean):,}")
display(reviews_clean[["document_id", "review_score", "review_creation_date", "review_comment_message"]].head())


Avaliações originais: 99,224
Avaliações com comentário válido: 40,950
Registros removidos por comentário vazio: 58,274


,document_id,review_score,review_creation_date,review_comment_message
0,e64fb393e7b32834bb789ff8bb30750e,5,2017-04-21,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,5,2018-03-01,Parabéns lojas lannister adorei comprar pela I...
2,8670d52e15e00043ae7de4c01cc2fe06,4,2018-05-22,aparelho eficiente. no site a marca do aparelh...
3,4b49719c8a200003f700d3d986ea1a19,4,2018-02-16,"Mas um pouco ,travando...pelo valor ta Boa."
4,3948b09f7c818e2d86c9a546758b2335,5,2018-05-23,"Vendedor confiável, produto ok e entrega antes..."


## 2. Enriquecimento relacional

O enriquecimento associa informações de pedido, cliente, itens, produto/categoria, vendedor e pagamento. Relações com múltiplos itens são agregadas por pedido.

In [3]:
prepared = enrich_reviews(raw)

print(f"Registros preparados: {len(prepared):,}")
print(f"IDs de avaliação distintos: {prepared['document_id'].nunique():,}")
display(prepared.head())


Registros preparados: 40,950
IDs de avaliação distintos: 40,950


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_id,order_status,order_purchase_timestamp,...,order_value,freight_value,product_category_name,product_category_name_english,seller_city,seller_state,payment_type,delivery_days,delivery_delay_days,document_id
0,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,de6dff97e5f1ba84a3cd9a3bc97df5f6,delivered,2017-04-09 17:41:13,...,179.99,42.85,ferramentas_jardim,garden_tools,ilicinea,MG,credit_card,10.644005,-19.619039,e64fb393e7b32834bb789ff8bb30750e
1,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,5986b333ca0d44534a156a52a8e33a83,delivered,2018-02-10 10:59:03,...,1199.00,134.25,esporte_lazer,sports_leisure,curitiba,PR,credit_card,18.232315,-8.310012,f7c4243c7fe1938f181bec41a392bdeb
2,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22,2018-05-23 16:45:47,a5224bdc7685fd39cd7a23404415493d,delivered,2018-05-14 10:29:02,...,586.56,26.69,eletroportateis,small_appliances,santo angelo,RS,credit_card,7.307755,-15.255417,8670d52e15e00043ae7de4c01cc2fe06
3,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.",2018-02-16,2018-02-20 10:52:22,8ebbeb09e94d1e789142983f73faa9cd,delivered,2018-02-12 16:07:34,...,35.00,7.78,beleza_saude,health_beauty,sao paulo,SP,credit_card,3.131481,-10.196597,4b49719c8a200003f700d3d986ea1a19
4,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23,2018-05-24 03:00:01,659ded3e9b43aaf51cf9586d03033b46,delivered,2018-05-18 18:20:45,...,89.99,13.07,informatica_acessorios,computers_accessories,araraquara,SP,credit_card,3.859745,-15.375845,3948b09f7c818e2d86c9a546758b2335


In [4]:
# Verifica duplicidade na tabela original
reviews = raw["reviews"]

print("=== TABELA ORIGINAL ===")
print(f"Total de avaliações: {len(reviews):,}")
print(f"Review IDs distintos: {reviews['review_id'].nunique():,}")
print(f"Review IDs duplicados: {reviews['review_id'].duplicated().sum():,}")

# Verifica duplicidade após o enriquecimento
print("\n=== BASE PREPARADA ===")
print(f"Total de registros: {len(prepared):,}")
print(f"Document IDs distintos: {prepared['document_id'].nunique():,}")
print(f"Document IDs duplicados: {prepared['document_id'].duplicated().sum():,}")

# Exibe exemplos dos IDs duplicados
duplicados = prepared[
    prepared["document_id"].duplicated(keep=False)
].sort_values("document_id")

display(
    duplicados[
        ["document_id", "review_id", "order_id", "review_score",
         "review_comment_message"]
    ].head(20)
)

=== TABELA ORIGINAL ===
Total de avaliações: 99,224
Review IDs distintos: 98,410
Review IDs duplicados: 814

=== BASE PREPARADA ===
Total de registros: 40,950
Document IDs distintos: 40,950
Document IDs duplicados: 0


,document_id,review_id,order_id,review_score,review_comment_message


In [5]:
duplicados_originais = (
    raw["reviews"][
        raw["reviews"]["review_id"].duplicated(keep=False)
    ]
    .sort_values("review_id")
)

display(duplicados_originais.head(20))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


## 3. Campos e valores ausentes após enriquecimento

Nem toda avaliação possui correspondência em todas as dimensões. Os valores ausentes são preservados como metadados nulos, sem inventar informação.

In [6]:
import pandas as pd

summary = pd.DataFrame({
    "tipo": prepared.dtypes.astype(str),
    "ausentes": prepared.isna().sum(),
    "ausentes_%": (prepared.isna().mean() * 100).round(2),
})
display(summary.sort_values("ausentes_%", ascending=False))


,tipo,ausentes,ausentes_%
review_comment_title,object,31120,76.00
order_delivered_customer_date,datetime64[ns],1876,4.58
delivery_days,float64,1876,4.58
delivery_delay_days,float64,1876,4.58
order_delivered_carrier_date,object,1197,2.92
product_id,object,524,1.28
seller_id,object,524,1.28
product_category_name,object,524,1.28
product_category_name_english,object,524,1.28
order_value,float64,524,1.28


## 4. Datas e indicadores de entrega

Os campos derivados permitem contextualizar relatos de experiência de compra. Valores nulos podem ocorrer quando o pedido não foi entregue ou não possui datas suficientes.

In [7]:
delivery_cols = [
    c for c in [
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "delivery_days", "delivery_delay_days"
    ] if c in prepared.columns
]
display(prepared[delivery_cols].describe(include="all").T)


,count,mean,min,25%,50%,75%,max,std
order_purchase_timestamp,40950,2017-12-25 20:55:46.917216,2016-09-04 21:15:19,2017-09-06 00:24:01.249999872,2018-01-15 16:25:54.500000,2018-04-29 11:49:31.750000128,2018-10-17 17:30:18,NaN
order_delivered_customer_date,39074,2018-01-10 14:24:27.951144192,2016-10-11 13:46:32,2017-09-19 22:39:48.500000,2018-01-31 19:13:10.500000,2018-05-11 18:53:11.500000,2018-10-02 00:18:50,NaN
order_estimated_delivery_date,40950,2018-01-19 02:33:54.989010944,2016-09-30 00:00:00,2017-09-27 00:00:00,2018-02-09 00:00:00,2018-05-23 00:00:00,2018-11-12 00:00:00,NaN
delivery_days,39074.0,13.495013,0.863426,6.941441,10.764149,16.743006,195.634016,10.810984
delivery_delay_days,39074.0,-10.701309,-134.30853,-16.314491,-11.585237,-6.227269,175.869109,11.361616


## 5. Construção dos documentos

Cada documento contém identificador, texto original limpo e metadados serializáveis.

In [8]:
documents = build_documents(prepared)

print(f"Documentos criados: {len(documents):,}")
display(documents[:3])
assert len(documents) == len(prepared)
assert len({doc["document_id"] for doc in documents}) == len(documents)


Documentos criados: 40,950


[{'document_id': 'e64fb393e7b32834bb789ff8bb30750e',
  'text': 'Recebi bem antes do prazo estipulado.',
  'metadata': {'review_id': 'e64fb393e7b32834bb789ff8bb30750e',
   'order_id': '658677c97b385a9be170737859d3511b',
   'review_score': '5',
   'review_creation_date': '2017-04-21T00:00:00',
   'review_answer_timestamp': '2017-04-21T22:02:06',
   'product_id': '52c80cedd4e90108bf4fa6a206ef6b03',
   'customer_id': 'de6dff97e5f1ba84a3cd9a3bc97df5f6',
   'customer_unique_id': 'c8cf6cb6b838dc7a33ed199b825e8616',
   'customer_city': 'gravatal',
   'customer_state': 'SC',
   'order_status': 'delivered',
   'order_purchase_timestamp': '2017-04-09T17:41:13',
   'order_delivered_customer_date': '2017-04-20T09:08:35',
   'order_estimated_delivery_date': '2017-05-10T00:00:00',
   'delivery_days': '10.64400462962963',
   'delivery_delay_days': '-19.619039351851853',
   'product_category_name': 'ferramentas_jardim',
   'product_category_name_english': 'garden_tools',
   'seller_id': 'a1043bafd471df

## 6. Persistência opcional da base preparada

O índice será reconstruído no próximo notebook. Este arquivo intermediário é útil para inspeção e auditoria; o pipeline oficial continua reproduzível a partir dos CSVs originais.

In [9]:
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
prepared_file = PROCESSED_DIR / "reviews_enriched.csv"
prepared.to_csv(prepared_file, index=False, encoding="utf-8")
print(f"Base enriquecida salva em: {prepared_file.relative_to(ROOT)}")


Base enriquecida salva em: data\processed\reviews_enriched.csv


## 7. Conclusões da preparação

Ao executar, verifique a quantidade de comentários válidos, a unicidade dos IDs, a cobertura dos metadados e a preservação de uma avaliação por documento.

**Próxima etapa:** [03 — Embeddings e indexação](03_embeddings_indexacao.ipynb).